# 03 · Memoria: continuidad y aprendizaje

Un LLM no recuerda nada. Cada llamada es un mundo nuevo, sin pasado. Cuando un chatbot
«se acuerda» de tu nombre, lo que ocurre en realidad es que **alguien le está reenviando
la conversación entera** en cada turno.

En LangGraph ese «alguien» tiene nombre y apellidos, y hay que distinguir **dos memorias
que se confunden constantemente**:

| | Memoria a corto plazo | Memoria a largo plazo |
|---|---|---|
| Qué guarda | la conversación en curso | hechos sobre el usuario |
| Dónde vive | `checkpointer` | `Store` |
| Se identifica por | `thread_id` (hilo) | *namespace* (usuario) |
| Sobrevive a | los turnos del hilo | los hilos **y** los procesos* |
| Analogía | lo que recuerdas de esta charla | lo que sabes de tu amigo |

\* si usas un `Store` persistente. El de esta libreta vive en memoria y muere con el
proceso; lo comentamos al final.

## 3.1 Sin checkpointer, el agente tiene amnesia

Empecemos por ver el problema, no la solución. Este grafo no tiene memoria: cada
`invoke` es una conversación nueva.

In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import SystemMessage
import comun

modelo = comun.llm()
PERSONALIDAD = "Eres un chef conciso y cercano."

def crear_grafo(checkpointer=None):
    """El mismo grafo, con o sin memoria, según lo que le pasemos."""
    def nodo(estado: MessagesState):
        mensajes = [SystemMessage(content=PERSONALIDAD)] + estado["messages"]
        return {"messages": [modelo.invoke(mensajes)]}

    c = StateGraph(MessagesState)
    c.add_node("agente", nodo)
    c.add_edge(START, "agente")
    c.add_edge("agente", END)
    return c.compile(checkpointer=checkpointer)

sin_memoria = crear_grafo()

print("Turno 1")
r1 = sin_memoria.invoke({"messages": [{"role": "user", "content": "Me llamo Luis y odio el cilantro."}]})
print("  ", r1["messages"][-1].content[:150])

print("\nTurno 2 (en teoría, la misma conversación)")
r2 = sin_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo?"}]})
print("  ", r2["messages"][-1].content[:200])

print(f"\nMensajes que ha visto el modelo en el turno 2: {len(r2['messages'])}")
print("El historial del turno 1 se ha tirado a la basura.")


Turno 1


   ¡Hola, Luis! No te preocupes, que en la cocina mandas tú. 

El cilantro es polémico (para algunos sabe a jabón), así que lo tachamos de la lista. Si a

Turno 2 (en teoría, la misma conversación)


   Eso no lo sé, ¡pero si me dices tu nombre, te saludaré como es debido mientras preparamos algo rico! 👨‍🍳

Mensajes que ha visto el modelo en el turno 2: 2
El historial del turno 1 se ha tirado a la basura.


## 3.2 Con checkpointer, hay hilo

Añadimos `InMemorySaver` al compilar y pasamos un `thread_id` en cada llamada.
**Ese identificador es la conversación.** Cambia el `thread_id` y cambias de
interlocutor.

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

con_memoria = crear_grafo(InMemorySaver())
cfg_A = {"configurable": {"thread_id": "alumno-A"}}

print("Turno 1")
r1 = con_memoria.invoke({"messages": [{"role": "user", "content": "Me llamo Luis y odio el cilantro."}]}, config=cfg_A)
print("  ", r1["messages"][-1].content[:150])

print("\nTurno 2")
r2 = con_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo? ¿Qué no me gusta?"}]}, config=cfg_A)
print("  ", r2["messages"][-1].content[:250])

print(f"\nMensajes que ha visto el modelo en el turno 2: {len(r2['messages'])}")
print("Aquí está la memoria: el historial crece y viaja en cada llamada.")


Turno 1


   ¡Hola, Luis! No te preocupes, es más común de lo que crees (a veces es genético).

Anotado: **Cero cilantro**. Si necesitas alguna receta o sustituto 

Turno 2


   Te llamas Luis y odias el cilantro. ¡Lo tengo clarísimo! 👨‍🍳

Mensajes que ha visto el modelo en el turno 2: 4
Aquí está la memoria: el historial crece y viaja en cada llamada.


## 3.3 Hilos distintos, memorias distintas

Esto es lo que hace útil al `thread_id`: aislamiento gratis entre conversaciones. Dos
alumnos, dos hilos, cero contaminación.

In [3]:
cfg_B = {"configurable": {"thread_id": "alumno-B"}}

print("Preguntamos lo mismo en el hilo B, que nunca ha oído hablar de Luis:")
r = con_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo?"}]}, config=cfg_B)
print("  ", r["messages"][-1].content[:250])


Preguntamos lo mismo en el hilo B, que nunca ha oído hablar de Luis:


   Eso no lo sé, ¡pero si me dices tu nombre, te saludaré como es debido mientras preparamos algo rico! 👨‍🍳


## 3.4 ¿Dónde está la memoria exactamente?

No hay magia ni «estado oculto del modelo». Hay una lista de mensajes guardada en un
`checkpoint`, y podemos abrirla. Merece la pena hacerlo una vez: desmitifica el concepto
para siempre.

In [4]:
estado = con_memoria.get_state(cfg_A)

print("Claves guardadas en el checkpoint:", list(estado.values.keys()))
print(f"Mensajes persistidos en el hilo alumno-A: {len(estado.values['messages'])}\n")
for i, mensaje in enumerate(estado.values["messages"]):
    texto = (mensaje.content or "").replace("\n", " ")[:70]
    print(f"  [{i}] {type(mensaje).__name__:18} {texto}")

print()
print("La memoria es esto: una lista que se guarda y se vuelve a cargar.")


Claves guardadas en el checkpoint: ['messages']
Mensajes persistidos en el hilo alumno-A: 4

  [0] HumanMessage       Me llamo Luis y odio el cilantro.
  [1] AIMessage          ¡Hola, Luis! No te preocupes, es más común de lo que crees (a veces es
  [2] HumanMessage       ¿Cómo me llamo? ¿Qué no me gusta?
  [3] AIMessage          Te llamas Luis y odias el cilantro. ¡Lo tengo clarísimo! 👨‍🍳

La memoria es esto: una lista que se guarda y se vuelve a cargar.


## 3.5 Memoria a largo plazo: el `Store`

El `checkpointer` resuelve «esta conversación». Pero hay cosas que no pertenecen a un
hilo concreto: que eres alérgico a los frutos secos, que prefieres recetas económicas,
que cocinas para cuatro. Eso debe sobrevivir a **cualquier** conversación.

Para eso está el `Store`: un almacén de clave-valor organizado por *namespaces*, pensado
para datos del usuario. Y aquí está la diferencia clave frente a meterlo en el prompt:

* el `Store` **no ocupa contexto** hasta que hace falta,
* se consulta **desde dentro de una herramienta**, cuando el agente lo necesita,
* y por tanto no se pierde al recortar el historial.

Dos herramientas: una que guarda y otra que lee. Fíjate en `runtime.store`.

In [5]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

ALUMNO = "luis"           # el namespace identifica al usuario, no al hilo
almacen = InMemoryStore()

@tool
def guardar_preferencia(clave: str, valor: str, runtime: ToolRuntime) -> str:
    """Guarda un dato del alumno (una alergia, una preferencia) para futuras conversaciones."""
    item = runtime.store.get((ALUMNO,), "preferencias")
    datos = dict(item.value) if item else {}
    datos[clave] = valor
    runtime.store.put((ALUMNO,), "preferencias", datos)
    return f"guardado: {clave} = {valor}"

@tool
def leer_preferencias(runtime: ToolRuntime) -> str:
    """Devuelve todos los datos guardados del alumno."""
    item = runtime.store.get((ALUMNO,), "preferencias")
    return str(item.value) if item else "no hay nada guardado"

chef = create_agent(
    modelo,
    tools=[guardar_preferencia, leer_preferencias],
    store=almacen,                        # <- aquí conectamos el almacén
    system_prompt=(
        "Eres un chef conciso. Cuando el alumno te cuente un dato personal "
        "relevante para cocinar, guárdalo con guardar_preferencia. "
        "Antes de proponer un plato, consulta leer_preferencias."
    ),
)

print("Almacén creado. Estado inicial:", almacen.get((ALUMNO,), "preferencias"))


Almacén creado. Estado inicial: None


## 3.6 La prueba que lo demuestra: memoria que cruza hilos

Guardamos en el **hilo 1** y preguntamos en el **hilo 2**, que es una conversación
completamente nueva. El `checkpointer` no puede ayudar aquí: los hilos no se hablan.
Si el agente lo recuerda, es porque lo ha leído del `Store`.

In [6]:
# Hilo 1: el alumno cuenta una alergia.
r1 = chef.invoke(
    {"messages": [{"role": "user", "content": "Soy alérgico a los frutos secos. Apúntalo."}]},
    config={"configurable": {"thread_id": "hilo-1"}},
)
print("HILO 1:", r1["messages"][-1].content[:200])
print("Almacén tras el hilo 1:", almacen.get((ALUMNO,), "preferencias").value)
print()

# Hilo 2: conversación nueva. ¿Se acuerda?
r2 = chef.invoke(
    {"messages": [{"role": "user", "content": "¿Qué alergias tengo? Propón una cena."}]},
    config={"configurable": {"thread_id": "hilo-2"}},
)
print("HILO 2:", r2["messages"][-1].content[:400])
print()
comun.ver_mensajes(r2)


HILO 1: Anotado. No incluiré frutos secos en ninguna de mis propuestas.
Almacén tras el hilo 1: {'alergias': 'frutos secos'}



HILO 2: Tienes alergia a los frutos secos.

Te propongo una cena: **Salmón al horno con espárragos y patatas baby**. Saludable, sencillo y seguro para ti.

[00] HumanMessage                 ¿Qué alergias tengo? Propón una cena.
[01] AIMessage  ->  TOOL_CALLS: leer_preferencias({}) 
[02] ToolMessage                  {'alergias': 'frutos secos'}
[03] AIMessage                    Tienes alergia a los frutos secos.  Te propongo una cena: **Salmón al horno con espárragos y patatas baby**. Saludable, sencillo y seguro para ti.


Y ahí está la distinción completa:

* El **hilo 2** no sabe nada del hilo 1 (`checkpointer`).
* Pero el agente **sí** conoce la alergia, porque la ha leído del `Store`.
* En la traza verás la `ToolMessage` con el contenido del almacén: el dato entró en la
  conversación **en el momento en que hacía falta**, y no antes.

## 3.7 Tres avisos que ahorran disgustos

**1. Olvidar el `checkpointer` no da error: no hace nada.** Es el fallo silencioso más
frecuente. Si tu agente parece no recordar, lo primero que hay que mirar es si compilaste
con `checkpointer=` y si estás pasando el mismo `thread_id`. Un `thread_id` nuevo es,
por definición, una conversación nueva.

**2. La memoria en RAM muere con el proceso.** `InMemorySaver` e `InMemoryStore` viven en
memoria: al reiniciar el cuaderno, todo desaparece. Para que sobreviva necesitas un
almacén persistente, por ejemplo SQLite:

```python
from langgraph.checkpoint.sqlite import SqliteSaver
# se instala aparte:  pip install langgraph-checkpoint-sqlite
with SqliteSaver.from_conn_string("memoria.db") as checkpointer:
    grafo = crear_grafo(checkpointer)
```

**3. El historial crece sin límite.** El turno 40 arrastra los 39 anteriores, y eso se
paga en tokens, en dinero y en latencia. Las soluciones habituales son **recortar** los
mensajes antiguos o **resumirlos**; LangChain 1.x trae middleware para ambas
(`SummarizationMiddleware`, `TrimMessagesMiddleware`). No lo usamos aquí para no
esconder la mecánica, pero es lo siguiente que hay que aprender.

---

## Resumen del módulo

| Idea | Detalle |
|---|---|
| Un LLM no recuerda: le reenvías el historial | La memoria es un dato, no una facultad |
| `checkpointer` + `thread_id` = memoria de conversación | Mismo id, misma conversación |
| Hilos distintos están aislados por construcción | Útil para multiusuario |
| La memoria es una lista de mensajes que se puede inspeccionar | `get_state(config)` |
| `Store` = memoria de largo plazo por usuario | Namespaces, no hilos |
| El `Store` se consulta desde herramientas y no ocupa contexto | Solo entra lo que se necesita |
| Sin `checkpointer`, la memoria falla **en silencio** | El error más caro del módulo |

## Ejercicios

1. **El mismo hilo, otro nombre.** En el apartado 3.2, pregunta «¿Cómo me llamo?» usando
   el `thread_id` `"alumno-B"`. ¿Por qué no lo sabe, si el grafo tiene memoria?

2. **Persistencia real.** Instala `langgraph-checkpoint-sqlite`, cambia `InMemorySaver`
   por `SqliteSaver` y reinicia el kernel. ¿Sobrevive la conversación de `alumno-A`?
   ¿Y el `InMemoryStore`? ¿Qué habría que cambiar para que el `Store` también persista?

3. **Namespace por alumno.** Modifica el `Store` para que el namespace sea
   `("alumnos", nombre)` en lugar de una constante. Hazlo con `context_schema` para no
   escribir el nombre a fuego. Después guarda datos de dos alumnos y comprueba que no se
   mezclan.

4. **El coste de la memoria.** Activa la memoria y mide cuántos mensajes viajan en el
   turno 5, el 10 y el 20. ¿Crece linealmente? ¿Qué pasa con la latencia? Escribe los
   números.

5. **Olvido selectivo.** Añade una herramienta `olvidar(clave)` que borre una preferencia
   del `Store`. Pídele al agente que olvide la alergia y comprueba que deja de aplicarla.

6. **Discusión para clase.** Una memoria que se equivoca es peor que no tener memoria:
   ¿qué pasa si el `Store` guarda «soy alérgico a los frutos secos» y el usuario luego
   dice «ya no lo soy»? ¿Quién decide cuándo el dato caduca? Diseña tu regla.